In [1]:
!uv add langchain_huggingface langchain-core requests

/bin/bash: line 1: uv: command not found


In [2]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")


## Tool Creation

In [3]:
# tool create

@tool
def multiply(a:int, b:int) -> int:
    """Given two number a and b and this tool return thier product"""
    return a*b

In [4]:
print(multiply.invoke({'a': 3, 'b': 5}))

15


In [5]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given two number a and b and this tool return thier product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Tool bind

In [6]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation"
)

llm = ChatHuggingFace(llm=llm)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
llm_with_tool = llm.bind_tools([multiply])

In [8]:
# llm

In [9]:
# llm_with_tool

## Tool Calling

In [10]:
llm_with_tool.invoke("Hi How are you")

AIMessage(content="Hi there! I'm doing great—thanks for asking. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 137, 'total_tokens': 178}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--6b23b51f-44a0-43c5-94f7-6a5e6f4574e7-0', usage_metadata={'input_tokens': 137, 'output_tokens': 41, 'total_tokens': 178})

In [11]:
query = HumanMessage('Multiply 3 with 9')

In [12]:
messages = [query]
messages

[HumanMessage(content='Multiply 3 with 9', additional_kwargs={}, response_metadata={})]

In [13]:
# llm_with_tool.invoke(messages)

In [14]:
# llm_with_tool.invoke("Can you multiply 3 with 9").tool_calls.tool_calls[0]
result = llm_with_tool.invoke(messages)

In [15]:
messages.append(result)

In [16]:
messages

[HumanMessage(content='Multiply 3 with 9', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":9}', 'name': 'multiply', 'description': None}, 'id': 'call_3SDKVCS6GbMIA1byPvZszsW3', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 139, 'total_tokens': 174}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--b9bc595d-6b10-4022-8124-611605ff39eb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 9}, 'id': 'call_3SDKVCS6GbMIA1byPvZszsW3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 139, 'output_tokens': 35, 'total_tokens': 174})]

In [17]:
result.tool_calls[0]['args']

{'a': 3, 'b': 9}

## Tool Execution

In [18]:
multiply.invoke(result.tool_calls[0]['args'])

27

In [19]:
tool_result = multiply.invoke(result.tool_calls[0])

In [20]:
messages.append(tool_result)

In [21]:
messages

[HumanMessage(content='Multiply 3 with 9', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":9}', 'name': 'multiply', 'description': None}, 'id': 'call_3SDKVCS6GbMIA1byPvZszsW3', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 139, 'total_tokens': 174}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--b9bc595d-6b10-4022-8124-611605ff39eb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 9}, 'id': 'call_3SDKVCS6GbMIA1byPvZszsW3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 139, 'output_tokens': 35, 'total_tokens': 174}),
 ToolMessage(content='27', name='multiply', tool_call_id='call_3SDKVCS6GbMIA1byPvZszsW3')]

In [22]:
# final_result = llm_with_tool.invoke(message)
final_result = llm_with_tool.invoke(messages).content

In [23]:
messages.append(final_result)

In [24]:
print(final_result)

The product of 3 and 9 is **27**.


In [25]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

clean_result = parser.invoke(final_result)
print(clean_result)


The product of 3 and 9 is **27**.
